# Quickstart — boto3

End-to-end walkthrough of AgentCore Memory using only the raw boto3 clients:

- `bedrock-agentcore-control` — control plane (`CreateMemory`, `UpdateMemory`, `DeleteMemory`)
- `bedrock-agentcore` — data plane (`CreateEvent`, `ListEvents`, `RetrieveMemoryRecords`)

The [CLI](./03-quickstart-cli.md) and [AgentCore SDK](./05-quickstart-agentcore-sdk.ipynb) quickstarts cover the same flow with different surfaces.

## Prerequisites

- AWS credentials for a region where AgentCore Memory is available.
- An IAM **memory execution role ARN** that AgentCore can assume for long-term extraction.
- Amazon Bedrock access to the embedding model used by semantic strategies.
- `pip install boto3`

In [ ]:
import os
import time
import uuid
from datetime import datetime, timezone

import boto3

REGION = os.getenv("AWS_REGION", "us-east-1")
MEMORY_ROLE_ARN = os.environ["MEMORY_EXECUTION_ROLE_ARN"]  # arn:aws:iam::<acct>:role/...
ACTOR_ID = "user-42"
SESSION_ID = f"sess-{int(time.time())}"

control = boto3.client("bedrock-agentcore-control", region_name=REGION)
data = boto3.client("bedrock-agentcore", region_name=REGION)

## 1. Create a memory resource

`CreateMemory` is asynchronous — the resource transitions `CREATING → ACTIVE`. Poll until it is ready before writing events.

In [ ]:
resp = control.create_memory(
    name="QuickstartMemory",
    description="Getting-started memory resource",
    eventExpiryDuration=30,
    memoryExecutionRoleArn=MEMORY_ROLE_ARN,
    clientToken=str(uuid.uuid4()),
)
memory_id = resp["memory"]["id"]
print("Created:", memory_id)

# Wait for ACTIVE
deadline = time.time() + 300
while time.time() < deadline:
    status = control.get_memory(memoryId=memory_id)["memory"]["status"]
    if status == "ACTIVE":
        break
    if status == "FAILED":
        raise RuntimeError("Memory creation failed")
    time.sleep(5)
print("Status:", status)

## 2. Write a short-term event

An event is a single turn (or set of turns) scoped to an `actorId` + `sessionId`.

In [ ]:
data.create_event(
    memoryId=memory_id,
    actorId=ACTOR_ID,
    sessionId=SESSION_ID,
    eventTimestamp=datetime.now(timezone.utc),
    payload=[
        {"conversational": {"role": "USER", "content": {"text": "My name is Alex and I prefer Python."}}},
        {"conversational": {"role": "ASSISTANT", "content": {"text": "Nice to meet you, Alex."}}},
    ],
)

## 3. Read events back

In [ ]:
events = data.list_events(memoryId=memory_id, actorId=ACTOR_ID, sessionId=SESSION_ID)["events"]
for e in events:
    print(e["eventId"], e["eventTimestamp"])

# Fetch one
if events:
    full = data.get_event(
        memoryId=memory_id,
        actorId=ACTOR_ID,
        sessionId=SESSION_ID,
        eventId=events[0]["eventId"],
    )
    print(full["event"]["payload"])

## 4. Add a built-in semantic strategy

A strategy tells AgentCore how to extract long-term records from events. Namespaces templated with `{actorId}` isolate records per user.

In [ ]:
control.update_memory(
    memoryId=memory_id,
    clientToken=str(uuid.uuid4()),
    memoryStrategies={
        "addMemoryStrategies": [
            {
                "semanticMemoryStrategy": {
                    "name": "UserFacts",
                    "namespaces": ["/users/{actorId}/facts"],
                }
            }
        ]
    },
)

# Extraction is asynchronous — give it ~60s before retrieving.
time.sleep(60)

## 5. Retrieve a memory record

In [ ]:
hits = data.retrieve_memory_records(
    memoryId=memory_id,
    namespace=f"/users/{ACTOR_ID}/facts",
    searchCriteria={"searchQuery": "What programming language does the user prefer?", "topK": 3},
)["memoryRecordSummaries"]

for h in hits:
    print(h["content"]["text"])

## 6. Teardown

In [ ]:
control.delete_memory(memoryId=memory_id, clientToken=str(uuid.uuid4()))

## See also

- [Concepts](./01-memory-concepts.md)
- Same flow in [CLI](./03-quickstart-cli.md) and [AgentCore SDK](./05-quickstart-agentcore-sdk.ipynb).